## IMPORTACION
Librerias, codigo, conexion

In [23]:
import mysql.connector
import pandas as pd
from datetime import datetime
import calendar
from dateutil.relativedelta import relativedelta
from pathlib import Path


In [52]:
from utils import (
    obtener_inicio_mes_siguiente, #Inicio-fin del mes siguiente
    ejecutar_y_guardar,
    obtener_mes_anterior,  #Inicio-fin mes
    guardar_csv 
)

In [25]:
conn = mysql.connector.connect(
    host = "datamart.mex.internal.lyftbikes.com",
    port =3306,
    database= "mex_datawarehouse_bss4",
    user= "dm_mex_ge",
    password= "m+y#J9JI9F+^4qjSJLu^R",
)

conn.cursor().execute(
    "SET SESSION max_execution_time = 300000"
)

## Vencimiento de membresías anuales


In [26]:
inicio_mes = obtener_inicio_mes_siguiente()

In [27]:
query_vencimientos =f"""
    SELECT
        member_accountNumber AS cuenta,
        accountHolderFirstName AS nombre,
        accountHolderLastName AS apellido,
        accountHolderEmail AS correo,
        CONVERT_TZ(
            FROM_UNIXTIME(end/1000,'%Y-%m-%d %H:%i:%s'),
            'UTC',
            'America/Costa_Rica'
        ) AS vencimiento

    FROM BikeSubscriptionFact s
    LEFT JOIN BikeAccountFact a
        ON s.member_accountnumber = a.publicAccountNumber

    WHERE (subscriptionType_id = 4 OR subscriptionType_id = 5)

      AND end/1000 BETWEEN
          UNIX_TIMESTAMP(
              CONVERT_TZ('{inicio_mes}','America/Costa_Rica','UTC')
          )

      AND UNIX_TIMESTAMP(
              CONVERT_TZ(
                  DATE_ADD('{inicio_mes}', INTERVAL 1 MONTH),
                  'America/Costa_Rica',
                  'UTC'
              )
          )

    ORDER BY vencimiento ASC
    """

In [28]:
ruta_archivo_vencimientos = (
    rf"C:\Users\tsunami.martinez\Downloads\Reportes"
    rf"\vencimientos_{inicio_mes[:7]}.csv"
)

In [29]:
df_vencimiento_membresias = ejecutar_y_guardar(
    query_vencimientos,
    conn,
    ruta_archivo_vencimientos
)

c:\Users\tsunami.martinez\Documents\Consultas SQL\utils.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


Archivo guadado: C:\Users\tsunami.martinez\Downloads\Reportes\vencimientos_2026-07.csv


## Reporte balanceo

In [30]:
fecha_inicio, fecha_fin = obtener_mes_anterior()

In [31]:
ruta_archivo_balanceo = (
    rf"C:\Users\tsunami.martinez\Downloads\Reportes"
        rf"\balanceo_{fecha_inicio[:7]}.csv")

In [32]:
query_balanceo = f"""
SELECT
    b.displayedNumber AS ID_bicicleta,

    CASE
        WHEN b.bikeModelName = 'CLASSIC'
        THEN 'Mecánica'
    END AS t_bicicleta,

    t.id AS ID_empleado,

    DATE_FORMAT(
        CONVERT_TZ(
            FROM_UNIXTIME(i.actionDateInMs/1000),
            'UTC',
            'America/Costa_Rica'
        ),
        '%d/%m/%Y'
    ) AS fech_inicio,

    DATE_FORMAT(
        CONVERT_TZ(
            FROM_UNIXTIME(f.actionDateInMs/1000),
            'UTC',
            'America/Costa_Rica'
        ),
        '%d/%m/%Y'
    ) AS fech_fin,

    DATE_FORMAT(
        CONVERT_TZ(
            FROM_UNIXTIME(i.actionDateInMs/1000),
            'UTC',
            'America/Costa_Rica'
        ),
        '%H:%i:%s'
    ) AS ini_viaj,

    DATE_FORMAT(
        CONVERT_TZ(
            FROM_UNIXTIME(f.actionDateInMs/1000),
            'UTC',
            'America/Costa_Rica'
        ),
        '%H:%i:%s'
    ) AS fin_viaj,

    ce_ini.logicalTerminal AS ciclo_inici,
    ce_fin.logicalTerminal AS ciclo_fin,

    SEC_TO_TIME(
        (f.actionDateInMs - i.actionDateInMs) / 1000
    ) AS tiemp_viaj

FROM BikeTechnicianActionFact AS i

INNER JOIN BikeTechnicianActionFact AS f
    ON i.bikeTransferByTechnicianId = f.bikeTransferByTechnicianId
    AND f.actionType_id = 0
    AND f.actionDateInMs > i.actionDateInMs

LEFT JOIN BikeTechnicianDim AS t
    ON i.tech_id = t.id

LEFT JOIN BikeDim AS b
    ON i.bike_id = b.id

LEFT JOIN BikeStationDim AS ce_ini
    ON i.station_id = ce_ini.id

LEFT JOIN BikeStationDim AS ce_fin
    ON f.station_id = ce_fin.id

WHERE i.actionType_id = 1

AND i.actionDateInMs >=
    UNIX_TIMESTAMP(
        CONVERT_TZ(
            '{fecha_inicio}',
            'America/Costa_Rica',
            'UTC'
        )
    ) * 1000

AND i.actionDateInMs <
    UNIX_TIMESTAMP(
        CONVERT_TZ(
            '{fecha_fin}',
            'America/Costa_Rica',
            'UTC'
        )
    ) * 1000

ORDER BY i.actionDateInMs
"""

In [33]:
df_vencimiento_membresias = ejecutar_y_guardar(
    query_balanceo,
    conn,
    ruta_archivo_balanceo
)

c:\Users\tsunami.martinez\Documents\Consultas SQL\utils.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


Archivo guadado: C:\Users\tsunami.martinez\Downloads\Reportes\balanceo_2026-05.csv


## Uso de sistema

In [34]:
ruta_archivo_uso_del_sistema = (
    rf"C:\Users\tsunami.martinez\Downloads\Reportes"
        rf"\Uso_del_sistema_{fecha_inicio[:7]}.csv")

In [35]:
query_uso_del_sistema_orig = f"""
    SELECT 
    v.member_accountNumber AS ID_usuario,
    b.displayedNumber AS ID_bicicleta,
    DATE_FORMAT(CONVERT_TZ(FROM_UNIXTIME(startTimeMs/1000, '%Y-%m-%d %H:%i:%s'), 'UTC', 'America/Costa_Rica' ), '%d, %m, %Y') AS fech_inicio,
    DATE_FORMAT(CONVERT_TZ(FROM_UNIXTIME(endTimeMs/1000, '%Y-%m-%d %H:%i:%s'), 'UTC', 'America/Costa_Rica' ), '%d, %m, %Y') AS fech_fin,
    DATE_FORMAT(CONVERT_TZ(FROM_UNIXTIME(startTimeMs/1000, '%Y-%m-%d %H:%i:%s'), 'UTC', 'America/Costa_Rica' ), '%H:%i:%s') AS ini_viaj,
    DATE_FORMAT(CONVERT_TZ(FROM_UNIXTIME(endTimeMs/1000, '%Y-%m-%d %H:%i:%s'), 'UTC', 'America/Costa_Rica' ), '%H:%i:%s') AS fin_viaj,
    ss.logicalTerminal AS ciclo_inici,
    es.logicalTerminal AS ciclo_fin,
    CONCAT(ss.longitude, ' ,', ss.latitude) AS ubica_inici,
    CONCAT(es.longitude, ' ,', es.latitude) AS ubica_fin,
    SEC_TO_TIME((endTimeMs - startTimeMs)/1000) AS tiemp_viaj,
    distanceInMeters AS dist_viaj,
    CONCAT(ss.longitude, " ", ss.latitude, ' ,', es.longitude, " ", es.latitude) AS GPS,
    CASE WHEN b.bikeModelName = 'CLASSIC' THEN 'Mecánica' ELSE NULL END AS t_unidad,
    CASE WHEN member_gender = 'M' THEN 'Hombre' 
        WHEN member_gender = 'F' THEN 'Mujer'
        WHEN member_gender = 'O' THEN 'Otro' 
        ELSE NULL
        END AS Género,
    FLOOR((UNIX_TIMESTAMP(CONVERT_TZ('{fecha_fin}' + INTERVAL 1 MONTH, 'America/Costa_Rica', 'UTC')) - member_birthday/1000) / (365*24*3600)) AS Edad

    FROM BikeRentalFact AS v
    LEFT JOIN BikeDim AS b
        ON v.bike_id = b.id
    LEFT JOIN BikeStationDim AS ss
        ON v.startStation_id = ss.id
    LEFT JOIN BikeStationDim AS es
        ON v.endStation_id = es.id

            
    WHERE v.endTimeMs >= UNIX_TIMESTAMP( CONVERT_TZ('{fecha_inicio}', 'America/Costa_Rica', 'UTC') ) * 1000
    AND v.endTimeMs < UNIX_TIMESTAMP( CONVERT_TZ('{fecha_fin}', 'America/Costa_Rica', 'UTC') ) * 1000 
                
            
    AND (totalDurationMs >= 120*1000
        OR startStation_id != endStation_id)
        """

In [36]:
df_uso_del_sistema = ejecutar_y_guardar(
    query_uso_del_sistema_orig,
    conn,
    ruta_archivo_uso_del_sistema
)

c:\Users\tsunami.martinez\Documents\Consultas SQL\utils.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


Archivo guadado: C:\Users\tsunami.martinez\Downloads\Reportes\Uso_del_sistema_2026-05.csv


## Tarjetas mi reporte uso 

In [42]:
ruta_archivo_tarjeta_mi = (
    rf"C:\Users\tsunami.martinez\Downloads\Reportes"
        rf"\tarjeta_mi_{fecha_inicio[:7]}.csv")

In [43]:
query_tarjeta_mi = f"""
SELECT 
	r.id AS VIAJE_ID, 
    acc.localizedValue0 AS ACCESO,
    CASE
		WHEN ISNULL(m.currentTransitCardNumber) THEN ""
        ELSE m.currentTransitCardNumber
	END  AS NUMERO_SERIE_HEX,
    stationOrig.name AS Ciclo_EstacionOrigen,
    DATE_FORMAT(CONVERT_TZ(FROM_UNIXTIME(r.startTimeMs/1000), 'UTC', 'America/Mexico_City'), '%Y-%m-%d %H:%i:%s') AS FECHA_HORA_TRANSACCION_SALIDA,
	stationDest.name AS Ciclo_EstacionArribo,
	DATE_FORMAT(CONVERT_TZ(FROM_UNIXTIME(r.endTimeMs/1000), 'UTC', 'America/Mexico_City'), '%Y-%m-%d %H:%i:%s') AS Fecha_Arribo,
    r.member_accountNumber AS customer,
    st.description_localizedValue1 AS membresia_tipo
FROM mex_datawarehouse_bss4.BikeRentalFact AS r
LEFT JOIN mex_datawarehouse_bss4.BikeDim AS b ON r.bike_id=b.id
LEFT JOIN mex_datawarehouse_bss4.BikeStationDim AS stationOrig ON r.startStation_id=stationOrig.id
LEFT JOIN mex_datawarehouse_bss4.BikeStationDim AS stationDest ON r.endStation_id=stationDest.id
LEFT JOIN mex_datawarehouse_bss4.BikeMemberFact AS m ON r.member_accountNumber=m.bikeMemberAttributes_accountNumber
LEFT JOIN mex_datawarehouse_bss4.RentalAccessMethodDim AS acc ON r.accessMethod_id=acc.id
LEFT JOIN mex_datawarehouse_bss4.BikeSubscriptionFact AS s ON r.subscriptionId=s.id
LEFT JOIN mex_datawarehouse_bss4.BikeSubscriptionTypeDim AS st ON s.subscriptionType_id=st.id

WHERE
	(r.startStation_id<>r.endStation_id OR billableDurationMs>=120000) AND
	r.endTimeMs>=UNIX_TIMESTAMP(CONVERT_TZ('{fecha_inicio}', 'America/Mexico_City', 'UTC'))*1000 AND # fecha de inicio 
	r.endTimeMs<=UNIX_TIMESTAMP(CONVERT_TZ('{fecha_fin}', 'America/Mexico_City', 'UTC'))*1000 # fecha de cierre
ORDER BY r.id ASC
;
"""

In [44]:
df_uso_tarjeta_mi = ejecutar_y_guardar(
    query_tarjeta_mi,
    conn,
    ruta_archivo_tarjeta_mi
)

c:\Users\tsunami.martinez\Documents\Consultas SQL\utils.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


Archivo guadado: C:\Users\tsunami.martinez\Downloads\Reportes\tarjeta_mi_2026-05.csv


In [45]:
df = pd.read_csv(
    r"C:\Users\tsunami.martinez\Downloads\Reportes\tarjeta_mi_2026-05.csv",
    encoding="utf-8"
)

In [48]:
origen = df[[
    "VIAJE_ID",
    "ACCESO",
    "NUMERO_SERIE_HEX",
    "customer",
    "Ciclo_EstacionOrigen",
    "FECHA_HORA_TRANSACCION_SALIDA"
]].copy()

origen = origen.rename(columns={
    "Ciclo_EstacionOrigen": "Ciclo_Estacion",
    "FECHA_HORA_TRANSACCION_SALIDA": "Fecha_Hora"
})

origen["Tipo"] = 70


arribo = df[[
    "VIAJE_ID",
    "ACCESO",
    "NUMERO_SERIE_HEX",
    "customer",
    "Ciclo_EstacionArribo",
    "Fecha_Arribo"
]].copy()

arribo = arribo.rename(columns={
    "Ciclo_EstacionArribo": "Ciclo_Estacion",
    "Fecha_Arribo": "Fecha_Hora"
})

arribo["Tipo"] = 71


df_final = pd.concat([origen, arribo], ignore_index=True)

In [51]:
df_final.head(2)

,VIAJE_ID,ACCESO,NUMERO_SERIE_HEX,customer,Ciclo_Estacion,Fecha_Hora,Tipo
0,64249816,Transit card,94BD1A16,GD24BN9S,CE-221 Emilio Castelar-Presidente Masaryk,2025-12-22 10:05:15,70
1,66390784,Mobile - QR code,953b8b2e,T252X3JD,CE-022 Reforma - Manchester,2026-01-30 20:37:14,70


In [54]:
guardar_csv(df_final,ruta_archivo_tarjeta_mi)

Archivo guadado: C:\Users\tsunami.martinez\Downloads\Reportes\tarjeta_mi_2026-05.csv


,VIAJE_ID,ACCESO,NUMERO_SERIE_HEX,customer,Ciclo_Estacion,Fecha_Hora,Tipo
0,64249816,Transit card,94BD1A16,GD24BN9S,CE-221 Emilio Castelar-Presidente Masaryk,2025-12-22 10:05:15,70
1,66390784,Mobile - QR code,953b8b2e,T252X3JD,CE-022 Reforma - Manchester,2026-01-30 20:37:14,70
2,67442531,Mobile - QR code,NaN,CY3FDX6G,CE-649 Héroes del 47 - Calzada de Tlalpan,2026-02-16 15:07:43,70
3,68684369,Transit card,C8518558,Q6UQ7MXS,CE-453 Pedregal-Teapa,2026-03-06 13:16:51,70
4,69316559,Mobile - QR code,NaN,57QCAM9R,CE-548 Dr. Mariano Azuela - Jose Antonio Alzate,2026-03-16 19:57:12,70
...,...,...,...,...,...,...,...
3230719,73953423,Transit card,94292E8A,MREYZUSC,CE-277 Prolongación Tonalá-Obrero Mundial,2026-05-31 23:58:13,71
3230720,73953429,Mobile - QR code,073A0991,635559,CE-366 Porfirio Díaz-Augusto Rodin,2026-05-31 23:55:03,71
3230721,73953434,Mobile - QR code,9596CAA2,QHECCTNV,CE-499 de los Maestros - Calzada de los Gallos,2026-05-31 23:59:56,71
3230722,73953442,Transit card,C85E9A53,552725,CE-377 Heriberto Frías-Miguel Laurent,2026-05-31 23:58:06,71
